# Prompt Evaluation and Iteration

Prompt engineering is not finished when a demo "looks good once." Production quality requires **measurement**: labeled test cases, comparable metrics, and a repeatable **iteration loop** when you change wording, examples, or model settings.

This notebook covers what to measure (accuracy, format compliance, latency, cost, consistency), how to build **test sets** and **golden prompts**, comparing prompt versions with A/B runs and rubrics, and a practical workflow: hypothesis → change → measure → document.

Live demonstrations evaluate two sentiment-classification prompts on the same reviews, track parse success, and sketch an iteration cycle you can extend in **10. Production Prompt Management**.

Prerequisites: **01–08** in this section (especially zero-shot/few-shot and structured output). Related: **11. GenAI Production Engineering → 03/01. Evaluating LLM Applications**.

---

## 1. What to Measure

### 1.1 Task Metrics

| Metric | Definition | When it matters |
|--------|------------|-----------------|
| **Accuracy** | % outputs matching gold labels or facts | Classification, extraction |
| **Format compliance** | % outputs parseable (JSON, enum, regex) | Pipelines feeding code |
| **Rubric score** | Human or LLM judge rates 1–5 on criteria | Summaries, creative, support replies |
| **Consistency** | Same input → same output across runs | Low temperature helps; still test |
| **Groundedness** | Claims supported by provided context | RAG, Q&A over docs |

### 1.2 Operational Metrics

| Metric | Definition | Why track |
|--------|------------|-----------|
| **Latency** | Time per API call (p50, p95) | UX, SLA |
| **Token usage** | Prompt + completion tokens | Cost forecasting |
| **Error rate** | API failures, timeouts, parse errors | Reliability |

A prompt that gains 2% accuracy but doubles tokens may be a bad trade for high-volume routes.

### 1.3 One Primary Metric per Task

Pick a **north-star** metric before iterating:

- Support routing → **category accuracy**
- JSON extraction → **schema-valid rate**
- User-facing summaries → **rubric score** + max length compliance

Secondary metrics guard against regressions (latency, cost, toxicity).

---

## 2. Test Sets and Golden Prompts

### 2.1 Building a Prompt Test Set

A **test set** is a table of inputs and expected outcomes (or evaluation criteria):

| Column | Purpose |
|--------|---------|
| `id` | Stable identifier for regression logs |
| `input` | User text or document snippet |
| `expected` | Gold label or reference answer (if exists) |
| `tags` | `edge_case`, `mixed_sentiment`, `long_input` |
| `notes` | Why this case matters |

Start with **20–50** real examples from logs or stakeholders. Grow when production surfaces failures.

### 2.2 Edge Cases and Regression Cases

| Category | Example |
|----------|---------|
| **Ambiguous** | Mixed positive/negative review → NEUTRAL |
| **Short** | "ok" |
| **Long** | Multi-paragraph email |
| **Adversarial** | Injection-like phrases (notebook 08) |
| **Format stress** | Input with JSON, emojis, typos |
| **Regression** | Bug that v3 fixed — must stay fixed in v4 |

Every production bug should become a **regression row** in the test set.

### 2.3 Golden Prompts

A **golden prompt** (or **prompt fixture**) is a versioned system + user template stored in git — not edited ad hoc in a notebook. Pair each golden prompt with:

- Model ID and inference settings (`temperature`, `max_tokens`)
- Expected metric thresholds (e.g. accuracy ≥ 0.85 on test set)
- Changelog entry when you promote a new version

---

## 3. Comparing Prompt Versions

### 3.1 A/B Testing Prompts

Run **the same test set** through prompt A and prompt B with **identical** model and settings (only the prompt text changes). Compare metrics side by side.

```
           ┌── Prompt A ──► metrics_A
Test set ──┤
           └── Prompt B ──► metrics_B
```

Statistical note: on small sets (n < 30), a 1–2 case swing moves accuracy a lot. Treat small gains cautiously; inspect **which** cases flipped.

### 3.2 Rubric Scoring (Human or LLM-as-Judge)

When there is no single correct string, use a **rubric**:

| Criterion | Score 1–5 |
|-----------|----------|
| Faithfulness to source | |
| Completeness | |
| Tone appropriate for audience | |

**LLM-as-judge** automates rough scoring — useful for development, not a full substitute for human review on high-stakes tasks. Use low temperature and show the rubric explicitly.

### 3.3 Diff Inspection

Beyond aggregate accuracy, list **cases that changed** between versions:

- Improved: v2 fixed false NEGATIVE on mixed review
- Regressed: v2 broke on short "ok" input

Ship only when net improvement on primary metric **and** no critical regressions.

---

## 4. Iteration Workflow

### 4.1 The Loop

1. **Baseline** — measure current prompt on test set
2. **Hypothesis** — "Adding one NEUTRAL few-shot example fixes mixed reviews"
3. **Change** — edit one variable (prompt text OR temperature, not both)
4. **Measure** — re-run full test set; log metrics
5. **Document** — changelog: what changed, numbers, decision promote/reject
6. **Repeat** or **promote** golden prompt version

### 4.2 Change One Thing at a Time

If you change prompt **and** model **and** temperature together, you cannot attribute results. Scientific iteration isolates variables.

### 4.3 When to Stop

- Primary metric meets threshold
- Diminishing returns (hours of work for 0.5% on tiny set)
- Cost/latency budget exceeded

Perfect prompts do not exist — **good enough with monitoring** beats endless offline tuning.

---

## 5. Demonstration Setup

Labeled sentiment test set (same spirit as notebook 03). Replace the placeholder API key before running.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import re
import time

from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)
MODEL = "gpt-4o-mini"
TEMPERATURE = 0.0

TEST_CASES = [
    {"id": "t1", "input": "Best purchase I made this year!", "expected": "POSITIVE", "tags": ["clear"]},
    {"id": "t2", "input": "Terrible quality, returned immediately.", "expected": "NEGATIVE", "tags": ["clear"]},
    {
        "id": "t3",
        "input": "Battery life is amazing but the screen is dim.",
        "expected": "NEUTRAL",
        "tags": ["mixed", "edge_case"],
    },
    {
        "id": "t4",
        "input": "It works. Shipping was normal.",
        "expected": "NEUTRAL",
        "tags": ["subtle"],
    },
]

VALID_LABELS = {"POSITIVE", "NEGATIVE", "NEUTRAL"}


def classify(review: str, system: str, user_template: str) -> tuple[str, float, int]:
    """Returns (raw_label, latency_sec, total_tokens)."""
    user = user_template.format(review=review)
    start = time.perf_counter()
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        temperature=TEMPERATURE,
        max_tokens=10,
    )
    elapsed = time.perf_counter() - start
    raw = (response.choices[0].message.content or "").strip().upper()
    tokens = response.usage.total_tokens if response.usage else 0
    # Extract label if model adds extra words
    for label in VALID_LABELS:
        if label in raw:
            return label, elapsed, tokens
    return raw, elapsed, tokens


print(f"Test set: {len(TEST_CASES)} cases")

---

## 6. Demonstration: Two Prompt Versions (A/B)

**v1** — vague instruction. **v2** — explicit labels, format rule, and one few-shot NEUTRAL example.

In [ ]:
PROMPT_V1 = {
    "name": "v1_vague",
    "system": "You classify customer review sentiment.",
    "user_template": "Review: {review}\nSentiment:",
}

PROMPT_V2 = {
    "name": "v2_structured_fewshot",
    "system": "Classify sentiment. Reply with exactly one word: POSITIVE, NEGATIVE, or NEUTRAL.",
    "user_template": """Examples:
Review: Loved the camera, hate the price. -> NEUTRAL

Review: {review}
Sentiment:""",
}


def evaluate_prompt(prompt: dict) -> list[dict]:
    rows = []
    for case in TEST_CASES:
        pred, latency, tokens = classify(
            case["input"], prompt["system"], prompt["user_template"]
        )
        rows.append(
            {
                "id": case["id"],
                "expected": case["expected"],
                "predicted": pred,
                "correct": pred == case["expected"],
                "format_ok": pred in VALID_LABELS,
                "latency_sec": round(latency, 3),
                "tokens": tokens,
            }
        )
    return rows


results_v1 = evaluate_prompt(PROMPT_V1)
results_v2 = evaluate_prompt(PROMPT_V2)

for label, rows in [("v1", results_v1), ("v2", results_v2)]:
    acc = sum(r["correct"] for r in rows) / len(rows)
    fmt = sum(r["format_ok"] for r in rows) / len(rows)
    avg_lat = sum(r["latency_sec"] for r in rows) / len(rows)
    avg_tok = sum(r["tokens"] for r in rows) / len(rows)
    print(f"=== {label} ===")
    print(f"Accuracy: {acc:.0%}  |  Format OK: {fmt:.0%}  |  Avg latency: {avg_lat:.2f}s  |  Avg tokens: {avg_tok:.0f}")
    for r in rows:
        mark = "OK" if r["correct"] else "MISS"
        print(f"  [{mark}] {r['id']}: expected={r['expected']} predicted={r['predicted']}")
    print()

Expect **v2** to improve mixed-review cases (t3) via the NEUTRAL few-shot example and stricter format. Inspect any case where v2 regresses — that blocks promotion.

---

## 7. Demonstration: Diff Between Versions

Highlight cases that **changed** prediction between v1 and v2.

In [ ]:
print("Cases where prediction changed:")
for c, r1, r2 in zip(TEST_CASES, results_v1, results_v2):
    if r1["predicted"] != r2["predicted"]:
        print(f"  {c['id']}: {r1['predicted']} -> {r2['predicted']} (expected {c['expected']})")
        print(f"    input: {c['input'][:60]}...")

print("\nRegressions in v2 (was correct in v1, wrong in v2):")
for r1, r2 in zip(results_v1, results_v2):
    if r1["correct"] and not r2["correct"]:
        print(f"  {r2['id']}: expected {r2['expected']}, got {r2['predicted']}")

---

## 8. Demonstration: Consistency Check

Run the same input multiple times. With `temperature=0`, outputs should be identical; with higher temperature, measure drift.

In [ ]:
REVIEW = TEST_CASES[2]["input"]
runs = []
for i in range(3):
    pred, _, _ = classify(REVIEW, PROMPT_V2["system"], PROMPT_V2["user_template"])
    runs.append(pred)

print("Review:", REVIEW)
print("Three runs at temperature=0:", runs)
print("All identical:", len(set(runs)) == 1)

---

## 9. Demonstration: LLM-as-Judge Rubric

For generative tasks without a single gold answer, score with an explicit rubric (development aid, not legal sign-off).

In [ ]:
SOURCE = TEST_CASES[2]["input"]
SUMMARY_A = "Great battery, bad screen."
SUMMARY_B = "The product has excellent battery life but the display quality is poor."

judge_prompt = f"""You evaluate summaries of a product review.

Review: {SOURCE}

Summary A: {SUMMARY_A}
Summary B: {SUMMARY_B}

Rubric (1-5 each): faithfulness, completeness, clarity.
Score both summaries. End with: WINNER: A or B"""

judge_out = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": judge_prompt}],
    temperature=0.0,
    max_tokens=250,
).choices[0].message.content

print(judge_out)

---

## 10. Demonstration: Iteration Log

Record decisions in a simple changelog structure (promote to JSON/YAML in production — notebook 10).

In [ ]:
def summarize_run(prompt_name: str, rows: list[dict]) -> dict:
    n = len(rows)
    return {
        "prompt": prompt_name,
        "accuracy": sum(r["correct"] for r in rows) / n,
        "format_compliance": sum(r["format_ok"] for r in rows) / n,
        "avg_tokens": sum(r["tokens"] for r in rows) / n,
    }


ITERATION_LOG = [
    {
        "version": "v1_vague",
        "hypothesis": "Baseline minimal instruction",
        "metrics": summarize_run("v1", results_v1),
        "decision": "reject — low accuracy on mixed reviews",
    },
    {
        "version": "v2_structured_fewshot",
        "hypothesis": "Explicit format + NEUTRAL few-shot fixes t3",
        "metrics": summarize_run("v2", results_v2),
        "decision": "promote if accuracy >= v1 and no regressions",
    },
]

for entry in ITERATION_LOG:
    m = entry["metrics"]
    print(f"{entry['version']}: acc={m['accuracy']:.0%} fmt={m['format_compliance']:.0%} tokens={m['avg_tokens']:.0f}")
    print(f"  Hypothesis: {entry['hypothesis']}")
    print(f"  Decision: {entry['decision']}\n")

---

## 11. Evaluation Checklist

Before shipping a new prompt version:

- [ ] Test set covers happy path + documented edge cases
- [ ] Same model and inference settings as production (unless testing model change)
- [ ] Primary metric meets threshold
- [ ] No critical regressions on diff review
- [ ] Format compliance acceptable for downstream parsers
- [ ] Token/latency within budget
- [ ] Changelog updated; golden prompt tagged in git
- [ ] Production monitoring planned for live drift

---

## 12. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Eyeballing one example | False confidence | Run full test set |
| Tiny test set | Noisy metrics | Add real failures |
| Changing many variables | Unknown cause | One change per iteration |
| No format metric | Silent pipeline breaks | Track parse success |
| LLM judge only | Bias, blind spots | Human spot-checks |
| No regression cases | Old bugs return | Add case per bug |

---

## 13. Summary

Evaluate prompts with **labeled test sets**, **format compliance**, **latency/tokens**, and **consistency**. Compare versions via A/B runs on the same inputs; inspect per-case diffs, not only aggregate accuracy. Use **rubrics** or LLM-as-judge when there is no single gold answer.

Iterate systematically: hypothesis → single change → measure → document → promote or reject. Pair offline evaluation with production monitoring.

Demonstrations compared v1 vs v2 sentiment prompts, diff analysis, consistency runs, rubric judging, and an iteration log.

Next: **10. Production Prompt Management** — versioning, templates, CI, and operating prompts in production.